<a href="https://colab.research.google.com/github/matthewpecsok/IS4490_student_course_files/blob/main/module-01-ai-llms-business-technology-ecosystem/module-01-lab-01-gemma-project-classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/matthewpecsok/IS4490-creation-fall2026/blob/main/module-01-ai-llms-business-technology-ecosystem/module-01-lab-01-gemma-project-classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


> ### Note on Labs and Assignments
>
> 🔧 Look for the **wrench emoji** — it marks code you must change. Routine run-only cells do not use it.
>
> 🖊 Look for the **writing emoji** — it marks analysis you must write.
>
> These sections are graded and are not optional.


* Make sure you are using a GPU runtime when you are running code that uses an LLM.

# Module 1 Lab 1: AI Project Classification and Decision Boundaries

**Notebook:** Student Template  
**Required model:** `gemma3:1b` through Ollama  
**Data:** Fictional cases only

This lab introduces the course's recurring assignment pattern: run a bounded AI task, preserve evidence, independently evaluate the output, and decide what responsibility must remain with a person.


## Learning Objectives

By completing this notebook, you will:

1. Classify business projects by their primary AI approach.
2. Evaluate model reasoning rather than treating it as an answer key.
3. Compare prompts that request unsupported precision or consequential authority.
4. Redesign AI use from decision making to evidence gathering.
5. Connect an AI proposal to a measurable outcome and simpler alternative.


## Important Instructions

1. Read the assignment and Module 1 reading first.
2. Use the fixed `gemma3:1b` model; do not substitute another model.
3. Change code where you see `🔧` and write analysis where you see `🖊`.
4. Run cells from top to bottom and preserve every output.
5. Use only the supplied fictional cases and résumé.
6. Restart the kernel and run all cells before submitting.

The notebook intentionally stops before a model run when required `TODO` code remains.


### TODO - IDENTIFY 🔧


In [ ]:
# 🔧 Replace "Your Name" before submitting.
STUDENT_NAME = "Your Name"
print(f"Student: {STUDENT_NAME}")


## Setup Ollama

Run the next two cells. The notebook connects to Ollama and downloads the required model if it is missing. The one-time `gemma3:1b` download is approximately 815 MB.


In [ ]:
# RUN THIS CELL
from datetime import datetime
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json
import textwrap

from IPython.display import Markdown, display

MODEL_NAME = "gemma3:1b"
OLLAMA_BASE_URL = "http://localhost:11434"
AUTO_PULL_MODEL = True
GENERATION_OPTIONS = {"temperature": 0, "seed": 4490, "num_ctx": 4096}
RUN_TIMESTAMP = datetime.now().astimezone().isoformat(timespec="seconds")

if STUDENT_NAME == "Your Name":
    print("TODO: Replace STUDENT_NAME before submitting.")

print(f"Required model: {MODEL_NAME}")
print(f"Run date: {RUN_TIMESTAMP}")


In [ ]:
# RUN THIS CELL
def require_finished(label, value):
    if value is None or "TODO" in str(value) or str(value).strip() == "Your Name":
        raise ValueError(f"Complete {label} before running this cell.")


def ollama_request(path, payload=None, timeout=120):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"{OLLAMA_BASE_URL}{path}",
        data=data,
        headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        details = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama returned HTTP {exc.code}: {details}") from exc
    except URLError as exc:
        raise RuntimeError(
            "Cannot connect to Ollama at http://localhost:11434. "
            "Install and start Ollama, then rerun this cell."
        ) from exc


def chat_once(prompt, timeout=600):
    response = ollama_request(
        "/api/chat",
        {
            "model": MODEL_NAME,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "options": GENERATION_OPTIONS,
        },
        timeout=timeout,
    )
    return response["message"]["content"].strip()


version_info = ollama_request("/api/version")
available_models = {
    item.get("name") or item.get("model")
    for item in ollama_request("/api/tags").get("models", [])
}
print(f"Connected to Ollama {version_info.get('version', 'unknown version')}.")

if MODEL_NAME not in available_models:
    if not AUTO_PULL_MODEL:
        raise RuntimeError(f"{MODEL_NAME} is not installed.")
    print(f"Downloading {MODEL_NAME}. This is a one-time download...")
    ollama_request("/api/pull", {"model": MODEL_NAME, "stream": False}, timeout=3600)

print(f"{MODEL_NAME} is ready.")


# Part 1: Classify Four AI Projects

Write one prompt that covers all four fictional retailer cases. The provided code uses a response schema only to keep the model output complete and tabular; it does not decide the classifications.


### TODO - INSTRUCT 🔧


In [ ]:
# 🔧 TODO - INSTRUCT
# Replace the TODO text with your own prompt. Your prompt must cover all four
# cases and request one primary category, any secondary categories, reasoning,
# and reasonable alternatives.
student_prompt = """TODO: Write your classification prompt here."""


In [ ]:
# RUN THIS CELL
require_finished("student name", STUDENT_NAME)
require_finished("classification prompt", student_prompt)

category_values = [
    "Rules and expert systems",
    "Predictive analytics and forecasting",
    "Classification and anomaly detection",
    "Recommendation and ranking",
    "Optimization",
    "Computer vision",
    "Speech AI",
    "Natural language processing",
    "Generative AI",
    "Agentic AI",
]
case_schema = {
    "type": "object",
    "properties": {
        "primary_category": {"type": "string", "enum": category_values},
        "secondary_categories": {
            "type": "array",
            "items": {"type": "string", "enum": category_values},
        },
        "reasoning": {"type": "string"},
        "reasonable_alternative": {"type": "string"},
    },
    "required": [
        "primary_category",
        "secondary_categories",
        "reasoning",
        "reasonable_alternative",
    ],
    "additionalProperties": False,
}
analysis_schema = {
    "type": "object",
    "properties": {key: case_schema for key in ["A", "B", "C", "D"]},
    "required": ["A", "B", "C", "D"],
    "additionalProperties": False,
}
response = ollama_request(
    "/api/chat",
    {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": student_prompt}],
        "stream": False,
        "format": analysis_schema,
        "options": GENERATION_OPTIONS,
    },
    timeout=600,
)
raw_classification_response = response["message"]["content"].strip()
analysis_data = json.loads(raw_classification_response)

display(Markdown("### Complete Classification Prompt\n```text\n" + student_prompt + "\n```"))
display(Markdown("### Complete Model Output\n```json\n" + raw_classification_response + "\n```"))


In [ ]:
# RUN THIS CELL
project_names = {
    "A": "A. Customer retention",
    "B": "B. Online merchandising",
    "C": "C. Warehouse quality",
    "D": "D. Employee support",
}

def markdown_cell(value):
    return str(value).replace("|", "\\|").replace("\n", " ").strip()

table_lines = [
    "| Project | AI's primary classification | Secondary classification, if any | Summary of AI's reasoning |",
    "|---|---|---|---|",
]
for key in "ABCD":
    result = analysis_data[key]
    secondary = ", ".join(result["secondary_categories"]) or "None identified"
    table_lines.append(
        "| "
        + " | ".join(
            markdown_cell(value)
            for value in [
                project_names[key],
                result["primary_category"],
                secondary,
                result["reasoning"],
            ]
        )
        + " |"
    )

classification_table_md = "\n".join(table_lines)
display(Markdown(classification_table_md))


### TODO - REFLECT 🖊

Write **2–4 sentences for each case**. Evaluate at least one specific model claim, use the input/task/output distinction, and challenge unsupported secondary categories.

#### A. Customer retention

🖊 TODO: State agree/partially agree/disagree and support your judgment.

#### B. Online merchandising

🖊 TODO: State agree/partially agree/disagree and support your judgment.

#### C. Warehouse quality

🖊 TODO: State agree/partially agree/disagree and support your judgment.

#### D. Employee support

🖊 TODO: State agree/partially agree/disagree and support your judgment.


# Part 2: Résumé Review and Decision Boundaries

The following job description and résumé are fictional. Run the two fixed prompts without changing them. The purpose is to observe how prompt framing can push a model toward unsupported precision or consequential authority—not to evaluate a real person.


In [ ]:
# RUN THIS CELL
JOB_DESCRIPTION = """Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience
"""

FICTIONAL_RESUME = """Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations
"""

display(Markdown("## Fictional Job Description\n```text\n" + JOB_DESCRIPTION + "\n```"))
display(Markdown("## Fictional Résumé\n```text\n" + FICTIONAL_RESUME + "\n```"))


## Compare Two Fixed Decision-Oriented Prompts

Both prompts use the same source material and model. Only the requested output changes.


In [ ]:
# RUN THIS CELL
shared_material = (
    "JOB DESCRIPTION\n---------------\n"
    + JOB_DESCRIPTION
    + "\n\nFICTIONAL RESUME\n------------------\n"
    + FICTIONAL_RESUME
)

score_prompt = (
    "Review this applicant for the job. Assign an overall fit score from 0 to 100 "
    "and explain the score.\n\n" + shared_material
)
decision_prompt = (
    "Review this applicant for the job. Make the final decision: HIRE or DO NOT HIRE. "
    "State the decision first, then explain it.\n\n" + shared_material
)

score_output = chat_once(score_prompt)
decision_output = chat_once(decision_prompt)

display(Markdown("### Prompt 1: 0–100 Score\n```text\n" + score_prompt + "\n```"))
display(Markdown("### Model Output 1\n" + score_output))
display(Markdown("### Prompt 2: Hire/Do Not Hire\n```text\n" + decision_prompt + "\n```"))
display(Markdown("### Model Output 2\n" + decision_output))


### TODO - REFLECT 🖊

Compare the two outputs.

🖊 TODO: Explain how the requested output format changed the response.

🖊 TODO: Identify evidence the model used, missed, assumed, or invented.

🖊 TODO: Explain why a 0–100 score can look more objective than the evidence supports.

🖊 TODO: Explain why neither output should transfer the employment decision to this model.

🖊 TODO: State what an accountable human reviewer must still do.


## Redesign the Task for Evidence Gathering

Use the same fictional materials, but constrain the model to organize evidence for an accountable human reviewer.


### TODO - INSTRUCT 🔧


In [ ]:
# 🔧 TODO - INSTRUCT
# Redesign the task so the model gathers job-relevant evidence without scoring,
# ranking, recommending, shortlisting, or making an employment decision.
evidence_prompt = """TODO: Write your evidence-gathering prompt here."""


In [ ]:
# RUN THIS CELL
require_finished("evidence-gathering prompt", evidence_prompt)
evidence_output = chat_once(evidence_prompt + "\n\n" + shared_material)
display(Markdown("### Evidence-Gathering Prompt\n```text\n" + evidence_prompt + "\n```"))
display(Markdown("### Evidence-Gathering Output\n" + evidence_output))


### TODO - REFLECT 🖊

🖊 TODO: Explain how the redesigned output differs from the two decision-oriented outputs.

🖊 TODO: Identify one useful evidence item and one item that still requires verification.

🖊 TODO: Explain how the redesign preserves human judgment and accountability.


# Part 3: Business Outcome and Simpler Alternative

Choose one Part 1 project. Start with the result the business needs, not the technology.


### TODO - DECIDE 🖊

Choose **one** project from Part 1.

**Selected project:** 🖊 TODO

**Business outcome:** 🖊 TODO: State the result that should improve.

**Metric, baseline, and target:** 🖊 TODO: Name one metric, the baseline evidence needed, and a proposed target.

**Simpler alternative:** 🖊 TODO: Identify a rule, conventional analysis, training intervention, or process change.

**Comparison evidence:** 🖊 TODO: Explain how the business could compare the simpler alternative with the AI proposal.

**Initial recommendation:** 🖊 TODO: Test the simpler approach first, pilot AI, combine them, or do not proceed—and explain why.


# Part 4: Reflect on the Assignment

Integrate what you observed across classification, résumé review, evidence gathering, and process fit.


### TODO - FINAL REFLECTION 🖊

Write **150–200 words** addressing all four questions:

1. What did the model do well and poorly in the classifications?
2. What did the résumé prompts reveal about unsupported precision or authority?
3. How would you use AI to gather evidence without delegating the employment decision?
4. Why compare an AI proposal with a simpler process change and measurable outcome?

🖊 TODO: Write your reflection here.


## Final Checklist

- [ ] I replaced `Your Name`.
- [ ] I used the required `gemma3:1b` model.
- [ ] I completed every `🔧` and `🖊` section.
- [ ] I preserved the four-case classification output.
- [ ] I evaluated all four classifications independently.
- [ ] I preserved both fixed résumé-prompt outputs.
- [ ] My evidence-gathering prompt avoids scoring, ranking, recommending, or deciding.
- [ ] My decision brief includes a metric, baseline, target, simpler alternative, and recommendation.
- [ ] My final reflection is 150–200 words.
- [ ] I restarted the kernel and ran all cells from top to bottom successfully.
- [ ] I used only the supplied fictional information.
